In [0]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from mlflow.models import infer_signature


FEATURE_TABLE = "workspace.aml_ml_3pct.customer_behavior_features"

FEATURE_COLUMNS = [
    "amount_deviation_ratio",
    "transaction_count_1h",
    "transaction_count_24h",
    "is_new_country",
    "distinct_countries_24h"
]

In [0]:
feature_sdf = spark.table(FEATURE_TABLE)

print(f"Feature table rows: {feature_sdf.count()}")

display(feature_sdf.limit(20))

In [0]:
model_pdf = (
    feature_sdf
    .select(
        "transaction_id",
        "generated_scenario",
        *FEATURE_COLUMNS
    )
    .toPandas()
)

print(f"Rows loaded for modelling: {len(model_pdf)}")

display(model_pdf.head(20))

In [0]:
X = (
    model_pdf[FEATURE_COLUMNS]
    .astype("float64")
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0.0)
)

print("Model features:")
print(X.columns.tolist())

print("\nMissing values:")
print(X.isnull().sum())

print("\nFeature statistics:")
display(X.describe())

In [0]:
model = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)

print(model)

In [0]:
model.fit(X)

raw_score = model.score_samples(X)

# Reverse the direction:
# higher value = more anomalous transaction
anomaly_score_raw = -raw_score

# Isolation Forest prediction:
#  1 = regular observation
# -1 = model outlier
model_prediction = model.predict(X)

print("Model trained successfully.")
print(f"Model outliers: {(model_prediction == -1).sum()}")

In [0]:
score_pdf = model_pdf[
    ["transaction_id", "generated_scenario"]
].copy()

score_pdf["anomaly_score_raw"] = anomaly_score_raw

score_pdf["ml_risk_score"] = (
    pd.Series(anomaly_score_raw)
    .rank(
        method="average",
        pct=True
    )
    .mul(100)
    .round(2)
)

score_pdf["model_outlier_flag"] = (
    model_prediction == -1
).astype(int)

display(
    score_pdf
    .sort_values(
        "ml_risk_score",
        ascending=False
    )
    .head(20)
)

In [0]:
result_pdf = (
    model_pdf
    .merge(
        score_pdf[
            [
                "transaction_id",
                "anomaly_score_raw",
                "ml_risk_score",
                "model_outlier_flag"
            ]
        ],
        on="transaction_id",
        how="inner"
    )
    .sort_values(
        "ml_risk_score",
        ascending=False
    )
)

display(result_pdf.head(30))

In [0]:
evaluation_pdf = score_pdf.copy()

evaluation_pdf["actual_anomaly"] = (
    evaluation_pdf["generated_scenario"] != "NORMAL"
).astype(int)

evaluation_pdf["predicted_anomaly"] = (
    evaluation_pdf["model_outlier_flag"] == 1
).astype(int)

In [0]:
true_positive = (
    (evaluation_pdf["actual_anomaly"] == 1)
    & (evaluation_pdf["predicted_anomaly"] == 1)
).sum()

false_positive = (
    (evaluation_pdf["actual_anomaly"] == 0)
    & (evaluation_pdf["predicted_anomaly"] == 1)
).sum()

false_negative = (
    (evaluation_pdf["actual_anomaly"] == 1)
    & (evaluation_pdf["predicted_anomaly"] == 0)
).sum()

true_negative = (
    (evaluation_pdf["actual_anomaly"] == 0)
    & (evaluation_pdf["predicted_anomaly"] == 0)
).sum()

print(f"True positives:  {true_positive}")
print(f"False positives: {false_positive}")
print(f"False negatives: {false_negative}")
print(f"True negatives:  {true_negative}")

In [0]:
precision = (
    true_positive
    / (true_positive + false_positive)
    if (true_positive + false_positive) > 0
    else 0
)

recall = (
    true_positive
    / (true_positive + false_negative)
    if (true_positive + false_negative) > 0
    else 0
)

print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")

In [0]:
display(
    spark.table("workspace.aml_ml.silver_transactions")
    .groupBy("generated_scenario")
    .count()
    .orderBy("generated_scenario")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


GOLD_TABLE = "workspace.aml_ml_3pct.gold_analyst_queue"


# Convert model scores from Pandas back to Spark.
score_sdf = spark.createDataFrame(
    score_pdf[
        [
            "transaction_id",
            "anomaly_score_raw",
            "ml_risk_score",
            "model_outlier_flag"
        ]
    ]
)


# Join transaction details and behavioural features
# with the Isolation Forest scores.
scored_transactions_df = (
    feature_sdf
    .drop("generated_scenario")
    .join(
        score_sdf,
        on="transaction_id",
        how="inner"
    )
)


# Rank only the transactions classified as outliers.
queue_window = Window.orderBy(
    F.desc("ml_risk_score"),
    F.desc("anomaly_score_raw")
)


gold_queue_df = (
    scored_transactions_df
    .filter(
        F.col("model_outlier_flag") == 1
    )
    .withColumn(
        "analyst_queue_rank",
        F.row_number().over(queue_window)
    )
    .withColumn(
        "scored_at",
        F.current_timestamp()
    )
    .select(
        "analyst_queue_rank",
        "transaction_id",
        "customer_id",
        "transaction_timestamp",

        "ml_risk_score",
        "anomaly_score_raw",
        "model_outlier_flag",

        "amount_eur",
        "currency",
        "country",
        "transaction_type",

        "amount_deviation_ratio",
        "transaction_count_1h",
        "transaction_count_24h",
        "is_new_country",
        "distinct_countries_24h",

        "scored_at"
    )
)


# Save the analyst queue as a managed Delta table.
(
    gold_queue_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)


print(f"Gold table created: {GOLD_TABLE}")
print(f"Transactions in analyst queue: {gold_queue_df.count()}")

display(
    gold_queue_df
    .orderBy("analyst_queue_rank")
)